# Stochastic Calculus, Step by Step · Lesson 14: The strange properties of Brownian paths

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. No slope anywhere · 3. First variation: infinitely long · 4. Quadratic variation: exactly T · 5. The multiplication table · 6. Average squares versus path squares · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 14,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. No slope anywhere

A derivative is the limit of the slope $(W_{t+h} - W_t)/h$ as $h \to 0$. For Brownian motion that slope is $N(0, h)/h$: normal with standard deviation $1/\sqrt h$, which **grows** as $h$ shrinks. The slope never settles: Brownian paths have no derivative, at any time.

In [ ]:
rng = np.random.default_rng(seed)
slope_rows = []
for h in (0.1, 0.01, 0.001, 0.0001):
    q = rng.normal(0, math.sqrt(h), 200_000) / h
    slope_rows.append({"h": h, "sd": float(q.std()), "sd_theory": 1 / math.sqrt(h),
                       "p_big": float((np.abs(q) > 10).mean()), "p_big_theory": 2 * (1 - N(10 * math.sqrt(h)))})
out["slope"] = {"rows": slope_rows}
pd.DataFrame(slope_rows)

## 3. First variation: infinitely long

Cut $[0, T]$ into $n$ steps and add up the **sizes** of the moves, $\sum|\Delta W_i|$. Each $|\Delta W_i|$ has mean $\sqrt{2\Delta t/\pi}$ (Lesson 4: $E|Z| = \sqrt{2/\pi}$), so the total has mean $\sqrt{2nT/\pi}$, which grows without limit. A Brownian path has **infinite length**. Below, one path is cut ever more finely (each grid contains the one before).

In [ ]:
T_ = 1.0
fine_n = 2**20
dW_fine = rng.normal(0, math.sqrt(T_ / fine_n), fine_n)
W_fine = np.concatenate([[0], np.cumsum(dW_fine)])
var_rows = []
for k_ in (4, 6, 8, 10, 12, 14, 16, 18, 20):
    n = 2**k_
    incr = np.diff(W_fine[:: fine_n // n])
    var_rows.append({"n": n, "first": float(np.abs(incr).sum()), "first_theory": math.sqrt(2 * n * T_ / math.pi),
                     "quad": float((incr**2).sum()), "quad_sd": math.sqrt(2 * T_**2 / n)})
out["var"] = {"rows": var_rows, "W_T": float(W_fine[-1])}
out["var"]["fig"] = {"k": [math.log2(x["n"]) for x in var_rows], "first": [x["first"] for x in var_rows],
                     "first_theory": [x["first_theory"] for x in var_rows], "quad": [x["quad"] for x in var_rows],
                     "T": [T_] * len(var_rows),
                     "ticks": [{"x": math.log2(x["n"]), "label": f"{x['n']:,}"} for x in var_rows[::2]]}
pd.DataFrame(var_rows)

## 4. Quadratic variation: exactly T

Add up the **squares** instead: $\sum(\Delta W_i)^2$ has mean $n \times \Delta t = T$ and variance $n \times 2\Delta t^2 = 2T^2/n \to 0$ (Lesson 6). So it converges to $T$: the **quadratic variation** $[W]_T = T$. The last two columns of the table above show it on one path. For a smooth path it is 0 instead: each move is of order $\Delta t$, its square of order $\Delta t^2$.

In [ ]:
smooth = S0 * np.exp(r * np.linspace(0, T_, fine_n + 1))           # a smooth path: cash growing at r
qv_smooth = {str(2**k_): float((np.diff(smooth[:: fine_n // 2**k_]) ** 2).sum()) for k_ in (4, 10, 16)}
logS = math.log(S0) + (r - 0.5 * sigma**2) * np.linspace(0, T_, fine_n + 1) + sigma * W_fine
qv_logS = {str(2**k_): float((np.diff(logS[:: fine_n // 2**k_]) ** 2).sum()) for k_ in (4, 8, 12, 16, 20)}
out["qv"] = {"smooth": qv_smooth, "logS": qv_logS, "sigma2T": sigma**2 * T_}
out["qv"]["rows"] = [{"n": int(k_), "rv": v} for k_, v in qv_logS.items()]
out["qv"]

## 5. The multiplication table

Over a small step, $\Delta W$ is of order $\sqrt{\Delta t}$. Adding up products over $[0, T]$:

| | $dt$ | $dW$ |
|---|---|---|
| $dt$ | 0 | 0 |
| $dW$ | 0 | $dt$ |

$\sum\Delta t^2 = T\Delta t \to 0$, $\;\sum\Delta W\,\Delta t \to 0$, $\;\sum\Delta W^2 \to T$. For two Brownian motions with correlation $\rho$, $dW\,dB = \rho\,dt$; independent ones give 0.

In [ ]:
table_rows = []
for k_ in (6, 10, 14, 18):
    n = 2**k_
    incr = np.diff(W_fine[:: fine_n // n]); dt_ = T_ / n
    table_rows.append({"n": n, "dt_dt": n * dt_**2, "dW_dt": float((incr * dt_).sum()), "dW_dW": float((incr**2).sum())})
rho = 0.6
B_incr = rho * dW_fine + math.sqrt(1 - rho**2) * rng.normal(0, math.sqrt(T_ / fine_n), fine_n)
indep = rng.normal(0, math.sqrt(T_ / fine_n), fine_n)
out["table"] = {"rows": table_rows, "rho": rho, "cross_rho": float((dW_fine * B_incr).sum()),
                "cross_indep": float((dW_fine * indep).sum())}
pd.DataFrame(table_rows)

## 6. Average squares versus path squares

Two statements look alike: $E[W_T^2] = T$, and $[W]_T = T$. They are very different. The first is an **average over many paths**: on any one path $W_T^2$ can be anything from 0 to large. The second holds **on every path**: add up the squared moves along any single path and you get $T$. Five paths:

In [ ]:
five = []
n = 2**16
for k_ in range(5):
    incr = rng.normal(0, math.sqrt(T_ / n), n)
    five.append({"path": k_ + 1, "W_T_sq": float(incr.sum() ** 2), "qv": float((incr**2).sum())})
out["five"] = five
pd.DataFrame(five)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** The slope of a Brownian path over a step $h$ is $Q_h = (W_{t+h} - W_t)/h$. Find its distribution, and compute $P(|Q_h| > 1)$ for $h = 0.01$ and for $h = 0.0001$.

**Exercise 2.** Show that $E\sum_{i=1}^n|\Delta W_i| = \sqrt{2nT/\pi}$ when $[0, T]$ is cut into $n$ equal steps. Evaluate it for $T = 1$ and $n = 100$, $10{,}000$ and $1{,}000{,}000$.

**Exercise 3.** Show that $Q_n = \sum_{i=1}^n(\Delta W_i)^2$ has mean $T$ and variance $2T^2/n$ (use $E[Z^4] = 3$), and use Chebyshev's inequality to show $Q_n \to T$.

**Exercise 4.** Show that $\sum\Delta W_i\,\Delta t$ has mean 0 and variance $T\Delta t^2$, and that $\sum\Delta t^2 = T\Delta t$. Why do both go to 0, while $\sum\Delta W_i^2$ does not?

**Exercise 5.** Find the quadratic variation over $[0, T]$ of (a) the smooth path $f(t) = 100e^{0.03t}$, (b) $\sigma W_t$, (c) $\ln S_t$ with $S_t = S_0e^{(r - \frac12\sigma^2)t + \sigma W_t}$.

**Exercise 6.** Let $B = \rho W + \sqrt{1 - \rho^2}\,W'$ with $W'$ a Brownian motion independent of $W$. Show that $B$ is a Brownian motion and that $\sum\Delta W_i\,\Delta B_i \to \rho T$.

**Exercise 7.** (Code) Simulate one year of daily log prices (252 steps) with $\sigma = 20\%$, and compute the **realised variance** $\sum(\Delta\ln S)^2$. Repeat 10,000 times: what are its mean and standard deviation, and how do they compare with $\sigma^2T$?

**Exercise 8.** (Code) On one path cut into $2^k$ steps for $k = 4, \dots, 20$, compute the first variation and the quadratic variation, and describe how each behaves as the number of steps grows.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")